# Język Python w Analizie Danych
# Działania na plikach

## Cel zajęć

- praktyczne opanowanie technik importowania danych z różnorodnych źródeł
- przećwiczenie pobierania danych z plików .csv, .xls i .html

## Część teoretyczna

### Dokumentacja
- Pandas: https://pandas.pydata.org/docs/
- Requests: https://requests.readthedocs.io/en/latest/

In [3]:
#instalacja modułów
#!pip install matplotlib seaborn

# Import modułów
import pandas as pd
import requests
from io import StringIO

### Przykłady

In [6]:
# Definicja adresu URL
url = "https://pl.wikipedia.org/wiki/Lista_pa%C5%84stw_%C5%9Bwiata"

# Nagłówki (User-Agent), aby udawać przeglądarkę i uniknąć blokady (Błąd 403)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.google.com/"
}

try:
    # Pobranie zawartości strony
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    # StringIO, aby przekazać tekst HTML do Pandas
    html_io = StringIO(response.text)
    tabele = pd.read_html(html_io)

    print(f"Liczba znalezionych tabel: {len(tabele)}")

    # Wybór pierwszej tabeli i podgląd
    df_panstwa = tabele[0]
    display(df_panstwa.head())

except Exception as e:
    print(f"Wystąpił błąd podczas pobierania danych: {e}")

Liczba znalezionych tabel: 10


,Lp.,Państwo,Mapa,Kontynent,Stolica,Powierzchnia (km²),Liczba ludności,Gęstość zaludnienia (os./km²)
0,1,Afganistan,NaN,Azja,Kabul,652 230,30 419 928[1],4664
1,2,Albania,NaN,Europa,Tirana,28 748,3 195 000[2],1111
2,3,Algieria,NaN,Afryka,Algier,2 381 741,34 178 188,1435
3,4,Andora,NaN,Europa,Andora,468,84 525,1806
4,5,Angola,NaN,Afryka,Luanda,1 246 700,17 312 000,1389


In [9]:
# Przykład wczytania lokalnego pliku
dane_html = pd.read_html('data/example_data_table.html')[0]
print(dane_html)

   id   name  age  score
0   1   John   21   85.0
1   2   Paul   20   83.4
2   3  Paula   16   68.6
3   4  Tomas   18   92.3
4   5  Diana   20   72.5


In [10]:
df = pd.read_xml("data/example_data_xml.xml")

print(df)

  przystanek    dzien   czas
0    Dworzec  roboczy  08:10
1    Dworzec  roboczy  08:30


### Zadanie 1
Załaduj zawartość pliku `google.csv` do obiektu DataFrame za pomocą biblioteki pandas i przypisz go do zmiennej google. Wydrukuj zawartość, podstawowe informacje i podstawowe statystyki zmiennej google w konsoli.

In [ ]:
# Task 1: wczytaj google.csv i pokaż podstawowe informacje
google = pd.read_csv('data/google.csv')
display(google.head())
print('--- info ---')
google.info()
print('--- describe ---')
display(google.describe(include='all'))

### Zadanie 2
a) We wczytanym obiekcie, zapisz obecne indeksy do nowej kolumny na początku tabeli.

b) Na końcu tabeli dodaj dwie nowe kolumny: `month` z numerem miesiąca i `year` zawierającą rok.

In [ ]:
# Task 2a: zapisz obecne indeksy do nowej kolumny na początku tabeli
# zakładamy, że zmienna `google` jest już wczytana
google.insert(0, 'orig_index', google.index)
display(google.head())

In [ ]:
# Task 2b: dodaj kolumny month i year (jeśli możliwe z kolumny z datą)
date_col = None
for c in ('Date','date','DATE','timestamp'):
    if c in google.columns:
        date_col = c
        break
if date_col is not None:
    dt = pd.to_datetime(google[date_col], errors='coerce')
    google['month'] = dt.dt.month
    google['year'] = dt.dt.year
else:
    google['month'] = pd.NA
    google['year'] = pd.NA
display(google.tail())

### Zadanie 3
Oblicz średnią cenę zamknięcia `Close` dla miesiąca (pogrupuj dane według miesiąca - kolumna `Month`, a następnie policz średnią wartość). Wydrukuj wynik do konsoli.

In [ ]:
# Task 3: oblicz średnią cenę zamknięcia 'Close' dla miesiąca
if 'Month' in google.columns:
    mean_close = google.groupby('Month')['Close'].mean()
else:
    # spróbuj wydobyć miesiąc z kolumny daty (jeśli istnieje)
    date_col = None
    for c in ('Date','date','DATE','timestamp'):
        if c in google.columns:
            date_col = c
            break
    if date_col is not None:
        google['Month'] = pd.to_datetime(google[date_col], errors='coerce').dt.month
        mean_close = google.groupby('Month')['Close'].mean()
    else:
        mean_close = pd.Series(dtype=float)
print(mean_close)

### Zadanie 4
Załaduj plik `data_tele.csv` do obiektu DataFrame o nazwie `df` z domyślnymi parametrami funkcji `pd.read_csv()` i wyświetl pierwsze 5 wierszy. Sprawdź występowanie pustych komórek. pogrupuj dane na poziomie zmiennych `Churn` oraz `PaymentMethod` i oblicz średnią wartość zmiennej `MonthlyCharges`. Wynik wydrukuj w konsoli.

In [ ]:
# Task 4: wczytaj data_tele.csv, pokaż head, sprawdź brakujące i policz średnie MonthlyCharges
df = pd.read_csv('data/data_tele.csv')
display(df.head())
print('Braki w kolumnach:')
print(df.isnull().sum())
res = df.groupby(['Churn','PaymentMethod'])['MonthlyCharges'].mean()
print(res)
display(res.reset_index(name='mean_MonthlyCharges'))

### Zadanie 5
Masz do dyspozycji plik `data_ex5.csv` zawierający dużą ilość danych. Wykonaj następujące operacje:

* Wczytaj dane w partiach po 1000 wierszy i dla każdej partii wybierz tylko te wiersze, w których wartość w kolumnie `one` jest większa niż 0.01 i mniejsza niż 0.5. Połącz otrzymane wyniki z wszystkich fragmentów w jeden DataFrame.
* Wczytaj dane w partiach po 500 wierszy i wybierz tylko te wiersze, w których wartość w kolumnie `four` jest mniejsza niż 0.01. Otrzymane wyniki zapisz do nowego pliku csv.
* Porównaj czas wykonania operacji wczytania pliku bez podziału na partie i z podziałem na partie.

In [ ]:
# Task 5: praca na dużym pliku z użyciem chunków i porównanie czasu
import time
# część 1: chunksize=1000, filtr na kolumnie 'one'
chunks = pd.read_csv('data/data_ex5.csv', chunksize=1000)
selected = []
t0 = time.time()
for ch in chunks:
    filt = ch[(ch['one'] > 0.01) & (ch['one'] < 0.5)]
    if not filt.empty:
        selected.append(filt)
res1 = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame()
t1 = time.time()
print('Chunked (1000) selection rows:', len(res1), 'time(s):', round(t1-t0,3))
# część 2: chunksize=500, filtr na kolumnie 'four' i zapis wyników
outfile = 'data/ex5_four_filtered.csv'
t0 = time.time()
first = True
for ch in pd.read_csv('data/data_ex5.csv', chunksize=500):
    filt2 = ch[ch['four'] < 0.01]
    if not filt2.empty:
        filt2.to_csv(outfile, mode='a', header=first, index=False)
        first = False
t1 = time.time()
print('Chunked (500) filtered and saved to', outfile, 'time(s):', round(t1-t0,3))
# część 3: porównanie z wczytaniem całości
t0 = time.time()
full = pd.read_csv('data/data_ex5.csv')
t1 = time.time()
print('Full load rows:', len(full), 'time(s):', round(t1-t0,3))

### Zadanie 6
Dany jest plik `data_test.csv`. Plik składa się z 5 kolumn, z czego ostatnia zawiera dwu lub trzy znakowy kod.
Wczytaj plik z użyciem pandas tak, aby wartości `NA` nie były traktowane jako braki (`NaN`). Wartościami pustymi są wyłącznie znaki: `#` i `-`.

Po wczytaniu wyświetl podstawowe informacje na temat datasetu. Policz wszystkie wartości puste w DataFrame i wyświetl wynik.

In [ ]:
# Task 6: wczytaj tak, aby 'NA' nie było traktowane jako brak; puste tylko '#' i '-'
df_test = pd.read_csv('data/data_test.csv', na_values=['#','-'], keep_default_na=False)
display(df_test.head())
print('Info:')
df_test.info()
print('Liczba wartości pustych (NaN):')
print(df_test.isnull().sum())

### Zadanie 7

Załaduj zawartość pliku `small.html` do obiektu DataFrame za pomocą biblioteki pandas i przypisz go do zmiennej `local_file`. Wydrukuj zawartość, podstawowe informacje i statystyki zmiennej `local_file` w konsoli. Wykonaj polecenia:

 - a) Zamień indeks podanego obiektu DataFrame na kolumnę `Data` i przypisz na stałe zmiany do zmiennej `local_file`.
  - b) Oblicz średnią cenę zamknięcia, otwarcia oraz różnicę między najwyższą a najniższą wartością dla każdego dnia.

**Pytanie:** Jaki błąd lub komunikat pojawi się, gdy spróbujesz odwołać się do klucza (etykiety), którego nie ma w serii?

In [ ]:
# Task 7: wczytaj small.html i pokaż podstawowe informacje
local_file = pd.read_html('data/small.html')[0]
display(local_file.head())
print('Info:')
local_file.info()

In [ ]:
# Task 7a: zamień indeks na kolumnę 'Data' i przypisz na stałe
local_file = local_file.reset_index()
# jeśli nazwa kolumny indeksu jest 'index', zmień na 'Data'
if 'index' in local_file.columns:
    local_file = local_file.rename(columns={'index':'Data'})
display(local_file.head())

In [ ]:
# Task 7b: oblicz średnie Close, Open oraz High-Low dla każdego dnia
key = 'Data' if 'Data' in local_file.columns else ('Date' if 'Date' in local_file.columns else local_file.columns[0])
group = local_file.groupby(key).agg({'Close':'mean','Open':'mean','High':'max','Low':'min'})
group['High-Low'] = group['High'] - group['Low']
display(group.head())
# Pytanie: odwołanie się do nieistniejącego klucza w serii -> KeyError

### Zadanie 8
Wybierz stronę Wikipedii zawierającą kilka rozbudowanych tabel (np. listę miast, listę państw, ranking). Wczytaj wszystkie tabele przy użyciu `pd.read_html()`, wybierz jedną, wykonaj podstawowe czyszczenie danych i zapisz wynik jako `cleaned_wiki.csv`. Określ liczbę brakujących elementów.

In [ ]:
# Task 8: wczytaj tabele z Wikipedii, oczyść i zapisz cleaned_wiki.csv
url = 'https://pl.wikipedia.org/wiki/Lista_pa%C5%84stw_%C5%9Bwiata'
tables = pd.read_html(url)
print('Znaleziono tabel:', len(tables))
wiki_df = tables[0].copy()
# proste czyszczenie: usunięcie białych znaków w tekstowych kolumnach
wiki_df = wiki_df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
out = 'data/cleaned_wiki.csv'
wiki_df.to_csv(out, index=False)
print('Zapisano do', out)
print('Liczba braków:', wiki_df.isnull().sum().sum())

### Zadanie 9
Ze strony [Dane gov - Rozkład jazdy](https://dane.gov.pl/pl/dataset/80/resource/1800,rozklad-jazdy-transportu-publicznego-xml/table) wybierz plik XML z informacjami o dowolnej linii transportu miejskiego. Utwórz DataFrame składający się z kolumn: `przystanek`, `dzien` i `czas odjazdu`.

In [ ]:
# Task 9: parsuj lokalny plik XML rozkładu (jeśli istnieje) do DataFrame
import xml.etree.ElementTree as ET
rows = []
xml_path = 'data/rozklad.xml'
try:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    # Przykładowa nawigacja: dostosuj do struktury pliku XML
    for stop in root.findall('.//stop'):
        name = stop.find('name').text if stop.find('name') is not None else None
        for day in stop.findall('.//day'):
            d = day.get('name') or (day.find('name').text if day.find('name') is not None else None)
            for time in day.findall('.//time'):
                rows.append({'przystanek': name, 'dzien': d, 'czas odjazdu': time.text})
    df_xml = pd.DataFrame(rows)
    display(df_xml.head())
except FileNotFoundError:
    print(f'Plik {xml_path} nie istnieje. Proszę podać właściwy plik XML.')
except Exception as e:
    print('Błąd podczas parsowania XML:', e)